## 🔥 <span style='text-decoration: double underline; color:rgb(10,110,217)'>**PyTorch: CNNs and Transfer Learning**</span>

### 📑  <span style='color:rgb(10,110,217)'><u>**Introduction**</u></span>

#### 👓 <span style='color:rgb(10,110,217)'><u>**Why Not Just Use an MLP for Images?**</u></span>

A fully connected MLP treats an image as a flat vector. A 28×28 grayscale image becomes 784 numbers, and a 224×224 RGB image becomes **150,528 numbers**. Connecting that to even a modest hidden layer explodes the number of parameters immediately.

More importantly, MLPs are **spatially blind** — they have no idea that pixel (10, 10) is next to pixel (10, 11). They treat every input independently, which means they cannot detect local patterns like edges, textures or shapes that can appear *anywhere* in the image.

Convolutional Neural Networks solve both problems.

#### 🪟 <span style='color:rgb(10,110,217)'><u>**The Convolution Operation**</u></span>

A convolution slides a small matrix called a **kernel** (or filter) across the image. At each position, it computes a dot product between the kernel weights and the underlying image patch. The result is a single number in the **feature map**.

<div align='center'>

![Convolution operation](https://upload.wikimedia.org/wikipedia/commons/1/19/2D_Convolution_Animation.gif)

</div>

The kernel slides across the image, computing a weighted sum at each position.

```
Input image:  (H, W)
Kernel:       (k, k)
Output:       (H - k + 1, W - k + 1)   ← without padding
              (H, W)                    ← with padding=1 and k=3
```

-  <span style='color:rgb(10,110,217)'>**Key parameters of `nn.Conv2d`**</span>

<div align='center'>

| Parameter | Meaning |
|-----------|---------|
| `in_channels` | Number of input channels (1 for grayscale, 3 for RGB) |
| `out_channels` | Number of filters to learn — each produces one feature map |
| `kernel_size` | Size of the sliding window (typically 3×3 or 5×5) |
| `stride` | How many pixels to skip between positions (default: 1) |
| `padding` | Zeros added around the border to preserve spatial size |

</div>

- <span style='color:rgb(10,110,217)'>**Why multiple filters?**</span>

Each filter learns to detect a **different pattern**: one might detect horizontal edges, another vertical edges, another corners. With `out_channels=32` you learn 32 different detectors simultaneously.

```python
# This learns 32 different 3×3 filters on a grayscale image
conv = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
# Parameters: 32 × (1 × 3 × 3) + 32 biases = 320
```
- <span style='color:rgb(10,110,217)'>**By how much is an image reduced between layers?**</span>

Each convolutional or pooling layer reduces the spatial dimensions of the input according to:

$$O = \left\lfloor \frac{I - F + 2P}{S} \right\rfloor + 1$$

Where $I$ is the input size, $F$ is the filter/kernel size, $P$ is the padding and $S$ is the stride. For example, a $32\times32$ image with a $3\times3$ kernel, no padding and stride 1 yields a $30\times30$ output. Stacking multiple layers without padding causes the spatial dimensions to shrink progressively, which is why **same padding** ($P = \lfloor F/2 \rfloor$) is often used to preserve them.

#### 🎯<span style='color:rgb(10,110,217)'><u>**Pooling — Downsampling Spatial Dimensions**</u></span>

After convolution, feature maps are typically large. **Pooling** reduces their spatial size while retaining the most important information. This makes the network faster, less prone to overfitting, and more robust to small translations in the image.

For example, **MaxPooling**  keeps the strongest activation in each region while **AveragePooling** takes the mean.

<div align='center'>

![MaxPooling vs AveragePooling](https://upload.wikimedia.org/wikipedia/commons/e/e9/Max_pooling.png)

</div>


#### 🏗️ <span style='color:rgb(10,110,217)'><u>**A Typical CNN Architecture**</u></span>

The building block of most CNNs is the pattern. 

```
[Conv2d → BatchNorm → ReLU → MaxPool]  ×  N  →  Flatten  →  Linear
```

<div align='center'>

![Typical CNN Architecture](https://imgs.search.brave.com/Gj98pMVd9k9vfFLZhedkr7WXlzRqAsB9OvuMdcEtmAI/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly91cGxv/YWQud2lraW1lZGlh/Lm9yZy93aWtpcGVk/aWEvY29tbW9ucy82/LzYzL1R5cGljYWxf/Y25uLnBuZw)

</div>


In this architectures, early layers detect simple features (edges) while deeper layers combine them into complex patterns (shapes, objects). In particular, as we go deeper:
- **Spatial dimensions shrink** (due to pooling)
- **Number of channels grows** (more abstract features)
- **Receptive field increases** (each neuron "sees" a larger area of the original image)

⚠️ **Warning:** this is the most common CNNs block, but you can play with the structure.


#### ⚖️ <span style='color:rgb(10,110,217)'><u>**Batch Normalization**</u></span>

Training deep networks is unstable — the distribution of activations shifts as parameters update, forcing later layers to constantly adapt. This is called **internal covariate shift**.

`BatchNorm2d` normalises the output of each layer across the batch to have mean ≈ 0 and std ≈ 1, then applies learnable scale (`γ`) and shift (`β`) parameters:

$$\hat{x} = \frac{x - \mu_{\text{batch}}}{\sqrt{\sigma^2_{\text{batch}} + \varepsilon}} \qquad \text{then} \qquad y = \gamma \cdot \hat{x} + \beta$$

- <span style='color:rgb(10,110,217)'>**Benefits in practice**</span>
     - **Faster convergence** — you can use larger learning rates
     - **Acts as regularisation** — reduces the need for Dropout
     - **Stabilises training** — gradients flow more consistently

```python
# Always place BatchNorm AFTER Conv and BEFORE activation
nn.Conv2d(32, 64, 3, padding=1),
nn.BatchNorm2d(64),   # ← normalises across the batch
nn.ReLU()
```

#### 🚚 <span style='color:rgb(10,110,217)'><u>**Transfer Learning and Fine Tuning**</u></span>

Training a deep CNN from scratch requires millions of labelled images and days of compute. **Transfer learning** lets you reuse a model that was already trained on a large dataset (ImageNet: 1.2M images, 1000 classes) and adapt it to your own problem.

The intuition: the early layers of a CNN trained on ImageNet learn universal features — edges, textures, colours — that are useful for *any* vision task. Only the last layers are task-specific.

These final layers are trained on the specific task. That is, fine tuning.

<div align='center'>

![Transfer Learning](https://imgs.search.brave.com/e8toumK87n7fcbiOmZfzgr7t5gDq2ogMVq0ucgQ9jko/rs:fit:860:0:0:0/g:ce/aHR0cHM6Ly9tZWRp/YS5nZWVrc2Zvcmdl/ZWtzLm9yZy93cC1j/b250ZW50L3VwbG9h/ZHMvMjAyNTAyMTMx/NTI3NTY3Mzg5NTgv/RmluZS1UdW5pbmcu/cG5n)

</div>

 -  <span style='color:rgb(10,110,217)'>**The two strategies**</span>

<div align='center'>

| Strategy | What you do | When to use |
|----------|------------|-------------|
| **Feature extraction** | Freeze all layers. Replace and train only the final classifier. | Small dataset, similar domain |
| **Fine-tuning** | Unfreeze some later layers. Train them with a small lr. | Medium dataset, or domain shift |

</div>

The example we will develop in this notebook consists of:

```python
# Step 1 — Load pretrained model
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')

# Step 2 — Freeze everything
for param in model.parameters():
    param.requires_grad = False

# Step 3 — Replace the classifier head
model.fc = nn.Linear(512, 10)  # only this layer trains

# Step 4 — Optionally unfreeze last block for fine-tuning
for param in model.layer4.parameters():
    param.requires_grad = True
```

#### ✂️<span style='color:rgb(10,110,217)'><u>**ResNets and Residual Connections**</u></span>

As networks get deeper, gradients tend to vanish before reaching the early layers — the **vanishing gradient problem**: 

During backpropagation, gradients are computed by repeatedly applying the chain rule through each layer:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial a_n} \cdot \frac{\partial a_n}{\partial a_{n-1}} \cdots \frac{\partial a_2}{\partial a_1} \cdot \frac{\partial a_1}{\partial w_1}$$

Each term in this product is a partial derivative of an activation function. With `sigmoid` or `tanh`, these derivatives are always **less than 1** — often much less. Multiply 50 numbers smaller than 1 together and the result is effectively zero. Early layers receive a gradient of ~0 and **stop learning entirely**.
```
Layer:      50   49   48  ...   3    2    1
Gradient:  0.9  0.8  0.7  ...  0.001  0.0001  ≈ 0  ← vanished
```

**ResNet** is an architecture that solved this in 2015 by introducing **skip connections** (also called residual connections). Here, instead of learning `F(x)`, the block learns the **residual** `F(x) + x`:

<div align='center'>
<img src="https://upload.wikimedia.org/wikipedia/commons/b/ba/ResBlock.png" alt="Residual Block" width="400">

</div>

Instead of learning the full transformation `H(x)`, the block only needs to learn the **residual** — the difference between the desired output and the identity:

$$H(x) = F(x) + x \implies F(x) = H(x) - x$$

This has two powerful consequences:

**1. Gradient highway** — the skip connection creates a direct path for gradients to flow backwards, bypassing the weight layers entirely. No matter how deep the network, the gradient can always travel through these shortcuts without being multiplied into oblivion:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial \text{output}} \cdot \underbrace{\left(\frac{\partial F(x)}{\partial x} + 1\right)}_{\geq 1 \text{ always}}$$

The `+1` ensures the gradient is never crushed to zero.

**2. Easier optimisation** — if a layer is not needed, it is much easier to push `F(x) → 0` (making the block an identity) than to learn `H(x) = x` from scratch through a stack of nonlinear transformations. The network can effectively *ignore* layers that do not help.

#### 🔎<span style='color:rgb(10,110,217)'><u>**Complementary Information: Data Augmentation**</u></span>

A model generalises better when it has seen more variation during training. **Data augmentation** applies random transformations to images *on the fly*, effectively multiplying the size of your dataset.

```python
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

```

> ⚠️ **Important:** augmentation is applied **only to the training set**. The validation and test sets use only `ToTensor()` + `Normalize()`.

#### 💻 <span style='color:rgb(10,110,217)'><u>**Moving Tensors to the Device**</u></span>

PyTorch can run computations on the CPU or on a GPU. To use the GPU, both the **model** and the **data** must live on the same device. The `.to(device)` method moves a tensor or a model to the specified device in-place.
```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SimpleCNN().to(device)       # move model weights to the device

images = images.to(device)           # move each batch of data too
labels = labels.to(device)
```

⚠️ **Common mistake:** forgetting to move the data. If the model is on the GPU but the input tensor is on the CPU, PyTorch will raise a `RuntimeError`. Both must be on the same device at all times.

- <span style='color:rgb(10,110,217)'>**Why does this matter?**</span>

<div align='center'>

| | CPU | GPU |
|---|---|---|
| Good for | Small models, debugging | Large models, matrix operations |
| Speed | Slow for deep learning | 10–100× faster for training |

</div>

$$
$$
> ⚠️ **Important:** `.to(device)` on a tensor returns a **new** tensor — it does not modify in-place. Always reassign: `x = x.to(device)`. On a model, it does modify in-place, so reassignment is optional but conventional.

---


###  🐉 <span style='color:rgb(10,110,217)'><u>**Level 3: Here be dragons**</u></span>
Use the [official documentation](https://docs.pytorch.org/docs/stable/index.html) while solving the exercises.

##### <span style='color:rgb(10,110,217)'>🔢 **Exercise 1: Build and Train a CNN on MNIST**</span>

> Build a CNN from scratch to classify MNIST handwritten digits. Define the architecture using two convolutional blocks followed by a linear classifier. Train it for **5 epochs** and report the test accuracy — aim for at least **98%**. Plot the training loss curve at the end.
>
> - Load MNIST with `torchvision.datasets.MNIST` and wrap it in a `DataLoader` with `batch_size=64`
> - Architecture: `Conv2d(1→32, k=3, p=1)` → `ReLU` → `MaxPool2d(2)` → `Conv2d(32→64, k=3, p=1)` → `ReLU` → `MaxPool2d(2)` → `Flatten` → `Linear(64·7·7, 128)` → `ReLU` → `Linear(128, 10)`
> - Loss: `nn.CrossEntropyLoss` — Optimizer: `Adam(lr=1e-3)`
> - Training loop: zero gradients → forward → loss → backward → step
> - Evaluation: compare `model(images).argmax(dim=1)` against `labels`

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt


torch.manual_seed(42)
################  --- FIRST TIME USING GPU!!! --- ################
# If the GPU is detected, device = 'cuda'. If not, device='cpu'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') 
print(f'Using device: {device}')

# ----- Data --------------------------------------------------------
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, 
                          batch_size= #TODO ,
                          shuffle= #TODO
                          )  # shuffle only for training!
test_loader  = DataLoader(test_dataset,  
                          batch_size= #TODO ,
                          shuffle= #TODO
                          )

# ----- Model -------------------------------------------------------- 
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=  #TODO ,
                      out_channels= #TODO ,
                      kernel_size=  #TODO ,
                      padding=      #TODO
                      ),
            #TODO: add ReLU ,
            #TODO: add MaxPool2d with kernel_size=2 ,
            nn.Conv2d(in_channels=   #TODO ,
                      out_channels=  #TODO ,
                      kernel_size= #TODO ,
                      padding=       #TODO
                      ), 
            #TODO: add ReLU
            #TODO: add MaxPool2d with kernel_size=2
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7, 128),  # hint: after two MaxPool2d(2), 28x28 becomes 7x7; we have 64 channels
            nn.ReLU(),
            nn.Linear(128, 10),  # hint: how many digit classes are there?
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = SimpleCNN().to(device)
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

# ----- Loss & optimizer --------------------------------------------------------
criterion =  #TODO # use CrossEntropyLoss
optimizer =  #TODO # use Adam with lr=1e-3

# ----- Training loop --------------------------------------------------------
train_losses = []
EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    model.train() # Important when dealing with MaxPool and BatchNormalization layers! (Is bestpractice to always write it)
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device) # .to(device) sends the tensor to the GPU (and all the calculations performed and to be perfomed with them)
        optimizer.   #TODO() # reset gradients
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.        #TODO() # backpropagation
        optimizer.   #TODO() # update weights
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

# ----- Evaluation --------------------------------------------------------
model.eval() 
correct = total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)   
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f'\nTest Accuracy: {100 * correct / total:.2f}%')

# ----- Loss curve --------------------------------------------------------
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o', color='steelblue', linewidth=2)
plt.title('Training Loss — CNN on MNIST')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

##### <span style='color:rgb(10,110,217)'>🔢 **Exercise 2: Transfer Learning on FashionMNIST with ResNet-18**</span>

> Load a pretrained `ResNet-18` and use it to classify FashionMNIST (10 clothing categories). **Freeze all existing layers** so their weights stay fixed, then replace only the final linear layer to output 10 classes. Train just that last layer for **3 epochs** and report the test accuracy.
>
> - Load `torchvision.models.resnet18(weights='IMAGENET1K_V1')` and freeze all parameters by setting `requires_grad = False`
> - Replace `model.fc` with a new `nn.Linear(512, 10)` — this is the only layer that will be trained
> - ResNet-18 expects 3-channel inputs: apply `Grayscale(num_output_channels=3)` and `Resize((64, 64))` in the transform
> - Use `Adam` with `lr=1e-3` — pass only `model.fc.parameters()` to the optimizer, not the whole model
> - After training, print how many parameters were trained vs. total (should be ~5 K out of 11 M)

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---  Data -------------------------------------------------------- 
# ResNet-18 expects 3-channel images. FashionMNIST is 1-channel 28x28,
# so we resize and replicate the channel.
transform = transforms.Compose([
    transforms.Resize((64, 64)),                         # upscale: 28x28 -> 64x64
    transforms.Grayscale(num_output_channels= #TODO ),     # 1 channel -> ? channels
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True,  download=True, transform=transform)
test_dataset  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

# ----- Load pretrained ResNet-18 -------------------------------------------------------- 
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')

# Freeze ALL layers: we want to reuse the learned features, not retrain them
for param in model.parameters():
    param.requires_grad =  #TODO # True or False?

# ----- Replace the classifier head --------------------------------------------------------
# model.fc currently outputs 1000 classes (ImageNet). We need 10.
# A new nn.Linear automatically has requires_grad=True.
model.fc = nn.Linear(# TODO , # TODO  )  # hint: ResNet-18 outputs 512 features

model = model.to(device)

# Verify: only model.fc should be trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} parameters')  # expect ~5K / 11M

# ----- Loss & optimizer --------------------------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model. #TODO ,  # which submodule to optimize?
                             lr=1e-3) # 

# ----- Training loop ------------------------------------------------------------------------
train_losses = []
EPOCHS = 3

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f'Epoch {epoch}/{EPOCHS}  |  Loss: {avg_loss:.4f}')

# ----- Evaluation ------------------------------------------------------------------------------
model.eval()
correct = total_samples = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct       += (preds == labels).sum().item()
        total_samples += labels.size(0)

print(f'\nTest Accuracy: {100 * correct / total_samples:.2f}%')

# ----- Loss curve -----------------------------------------------------------------------------------------------------
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), train_losses, marker='o', color='darkorange', linewidth=2)
plt.title('Training Loss — ResNet-18 Transfer Learning on FashionMNIST')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.xticks(range(1, EPOCHS + 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---

### 🎌<span style='color:rgb(10,110,217)'><u>**Summary**</u></span>

This notebook covers **CNNs and Transfer Learning in PyTorch** through 10 hands-on exercises. It starts by motivating CNNs over MLPs for image data, then progressively builds intuition around convolutions, pooling, and shape tracking on MNIST. It then introduces **Data Augmentation** and **Batch Normalization**, and closes with Transfer Learning and Fine-Tuning using a pretrained **ResNet-18** on FashionMNIST — culminating in a full comparison between models trained from scratch vs. pretrained.